# Receptor Mapping — Working Memory Pipeline

Pipeline mapping PET-derived neuroreceptor densities and
T1w/T2w myelination onto working memory regions defined by Neurosynth, parcellated
at the Schaefer-100 atlas level.

## Pipeline overview
1. **Data Processing** — binarise the Neurosynth NIfTI map and parcellate all features.
2. **Visualize on Schaefer** — surface plots of WM regions and receptor densities.
3. **Multicollinearity Analysis** — VIF decomposition and correlation heatmap.
4. **PLS-DA Analysis** — latent variable analysis + spin permutation testing.
5. **Logistic Regression Analysis** — forward selection and spin-tested model comparison.

## Imports

In [3]:
import sys
import os

sys.path.insert(0, os.path.abspath("."))

from Scripts.Data_Processing import binarize_nifti, process_neuro_receptor_data
from Scripts.Visualize_On_Schaefer import visualize_WorkingMemory, visualize_receptor_densities
from Scripts.Multicollinearity_Analysis import calculate_multicollinearity
from Scripts.PLSDA_Analysis import plsda_analysis
from Scripts.LogisticRegression_Analysis import run_forward_selection_with_baseline, run_spin_absolute_significance

---
## 1. Data Processing
### 1a. Binarize the Neurosynth map at vertex level

In [4]:
# Configuration
INPUT_NII   = "Data/Raw/Neurosynth/working_memory_association-test_z_FDR_0.01.nii"
OUTPUT_NII  = "Data/Processed/Neurosynth/working_memory_association-test_z_FDR_0.01_binary.nii"
THRESHOLD   = 0          # Voxels with value > THRESHOLD become 1, rest become 0

# Run
binarize_nifti(
    input_nii=INPUT_NII,
    output_nii=OUTPUT_NII,
    threshold=THRESHOLD,
)

parsed '(x > 0)' as 'x > 0'


### 1b. Parcellate receptor, T1w/T2w, and working memory data onto Schaefer-100

In [5]:
# Configuration 
WB_VIEW_PATH          = "/Applications/wb_view.app/Contents/MacOS"
WM_MAP_PATH           = "Data/Processed/Neurosynth/working_memory_association-test_z_FDR_0.01_binary.nii"
RECEPTOR_NAMES_PATH   = "Data/Raw/Hansen_2022/receptor_names_pet.csv"
RECEPTOR_DATA_PATH    = "Data/Raw/Hansen_2022/receptor_data_scale100.csv"
T1WT2W_DF_PATH        = "Data/Raw/Dorcioman_2026/schaefer100_t1wt2w.csv"
OUTPUT_DIRECTORY_PATH = "Data/Processed/ReceptorSelection"

RECEPTOR_DF_NAME         = "Receptor_Schaefer100_df.csv"
T1WT2W_DF_NAME           = "T1wT2W_Schaefer100_df.csv"
T1WT2W_RECEPTOR_DF_NAME  = "T1wT2WReceptor_Schaefer100_df.csv"
WM_PARC_NAME             = "WorkingMemory_Schaefer100.csv"

# Binary threshold values applied to the continuous WM parcellation score
THRESHOLDS = [0, 0.001, 0.01, 0.1]

process_neuro_receptor_data(
    wb_view_path=WB_VIEW_PATH,
    wm_map_path=WM_MAP_PATH,
    receptor_names_path=RECEPTOR_NAMES_PATH,
    receptor_data_path=RECEPTOR_DATA_PATH,
    T1wT2W_df_path=T1WT2W_DF_PATH,
    output_directory_path=OUTPUT_DIRECTORY_PATH,
    receptor_df_name=RECEPTOR_DF_NAME,
    T1wT2W_receptor_df_name=T1WT2W_RECEPTOR_DF_NAME,
    wm_parc_name=WM_PARC_NAME,
    T1wT2W_df_name=T1WT2W_DF_NAME,
    thresholds=THRESHOLDS,
)

Please cite the following papers if you are using this function:
  [primary]:
    Alexander Schaefer, Ru Kong, Evan M Gordon, Timothy O Laumann, Xi-Nian Zuo, Avram J Holmes, Simon B Eickhoff, and BT Thomas Yeo. Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity mri. Cerebral cortex, 28(9):3095–3114, 2018.
Dataset atl-schaefer2018 already exists. Skipping download.
Processing complete.
WM continuous + binary thresholds saved in: WorkingMemory_Schaefer100.csv
Thresholds used: [0, 0.001, 0.01, 0.1]


---
## 2. Visualize on Schaefer Parcellation
### 2a. Working memory areas

In [6]:
# Configuration
WM_CSV_PATH  = "Data/Processed/ReceptorSelection/WorkingMemory_Schaefer100.csv"
WM_FIGS_DIR  = "Figures/WorkingMemory_Parc"

# Run
visualize_WorkingMemory(
    wm_csv_path=WM_CSV_PATH,
    output_dir=WM_FIGS_DIR,
)

Please cite the following papers if you are using this function:
  [primary]:
    Alexander Schaefer, Ru Kong, Evan M Gordon, Timothy O Laumann, Xi-Nian Zuo, Avram J Holmes, Simon B Eickhoff, and BT Thomas Yeo. Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity mri. Cerebral cortex, 28(9):3095–3114, 2018.
Dataset atl-schaefer2018 already exists. Skipping download.
Saved: Figures/WorkingMemory_Parc/WorkingMemory_Score.png
Saved: Figures/WorkingMemory_Parc/Thr_0.png
Saved: Figures/WorkingMemory_Parc/Thr_0p001.png
Saved: Figures/WorkingMemory_Parc/Thr_0p01.png
Saved: Figures/WorkingMemory_Parc/Thr_0p1.png


### 2b. Receptor densities and T1w:T2w

In [7]:
# Configuration
RECEPTOR_CSV_PATH  = "Data/Processed/ReceptorSelection/T1wT2WReceptor_Schaefer100_df.csv"
RECEPTOR_FIGS_DIR  = "Figures/Receptor_T1wT2w_Parc"

# Set to None to plot all receptors in the CSV
RECEPTORS_TO_PLOT  = ["D1", "D2", "DAT", "NET", "NMDA", "GABAa", "T1w:T2w"]

# Run
visualize_receptor_densities(
    receptor_csv_path=RECEPTOR_CSV_PATH,
    output_dir=RECEPTOR_FIGS_DIR,
    receptors=RECEPTORS_TO_PLOT,
)

Please cite the following papers if you are using this function:
  [primary]:
    Alexander Schaefer, Ru Kong, Evan M Gordon, Timothy O Laumann, Xi-Nian Zuo, Avram J Holmes, Simon B Eickhoff, and BT Thomas Yeo. Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity mri. Cerebral cortex, 28(9):3095–3114, 2018.
Dataset atl-schaefer2018 already exists. Skipping download.
Saved: Figures/Receptor_T1wT2w_Parc/D1.png
Saved: Figures/Receptor_T1wT2w_Parc/D2.png
Saved: Figures/Receptor_T1wT2w_Parc/DAT.png
Saved: Figures/Receptor_T1wT2w_Parc/NET.png
Saved: Figures/Receptor_T1wT2w_Parc/NMDA.png
Saved: Figures/Receptor_T1wT2w_Parc/GABAa.png
Saved: Figures/Receptor_T1wT2w_Parc/T1w_T2w.png


---
## 3. Multicollinearity Analysis

In [8]:
# Configuration
X_PATH_MULTI        = "Data/Processed/ReceptorSelection/Receptor_Schaefer100_df.csv"
MULTI_OUTPUT_DIR    = "Results/Multicollinearity"
MULTI_FIGURES_DIR   = "Figures/Multicollinearity"

# Receptors to include
SELECTED_RECEPTORS = [
    "5HT1a", "5HT1b", "5HT2a", "5HT4", "5HT6",
    "A4B2", "M1",
    "D1", "D2",
    "mGluR5", "NMDA",
    "CB1", "GABAa", "H3", "MOR",
    "5HTT", "DAT", "VAChT", "NET"
    ]

# Run
calculate_multicollinearity(
    X_path=X_PATH_MULTI,
    output_directory_path=MULTI_OUTPUT_DIR,
    figure_directory_path=MULTI_FIGURES_DIR,
    selected_receptors=SELECTED_RECEPTORS,
)

CSV outputs saved to:  Results/Multicollinearity/AllReceptors
Heatmap saved to:      Figures/Multicollinearity/AllReceptors


---
## 4. PLS-DA Analysis

In [9]:
# Configuration
X_PATH_PLSDA        = "Data/Processed/ReceptorSelection/Receptor_Schaefer100_df.csv"
Y_PATH_PLSDA        = "Data/Processed/ReceptorSelection/WorkingMemory_Schaefer100.csv"
PLSDA_OUTPUT_DIR    = "Results/PLSDA"
PLSDA_FIGURES_DIR   = "Figures/PLSDA"

THRESHOLD_COLUMNS   = ["Thr_0", "Thr_0p1", "Thr_0p01", "Thr_0p001"]
PLSDA_N_PERM        = 1000
PLSDA_SEED          = 1234

SELECTED_RECEPTORS = [
    "5HT1a", "5HT1b", "5HT2a", "5HT4", "5HT6",
    "A4B2", "M1",
    "D1", "D2",
    "mGluR5", "NMDA",
    "CB1", "GABAa", "H3", "MOR",
    "5HTT", "DAT", "VAChT", "NET"
    ]

plsda_analysis(
    X_path=X_PATH_PLSDA,
    Y_path=Y_PATH_PLSDA,
    output_directory_path=PLSDA_OUTPUT_DIR,
    figure_directory_path=PLSDA_FIGURES_DIR,
    number_permutations=PLSDA_N_PERM,
    seed=PLSDA_SEED,
    threshold_columns=THRESHOLD_COLUMNS,
    selected_receptors=SELECTED_RECEPTORS,
)

Please cite the following papers if you are using this function:
  [primary]:
    Alexander Schaefer, Ru Kong, Evan M Gordon, Timothy O Laumann, Xi-Nian Zuo, Avram J Holmes, Simon B Eickhoff, and BT Thomas Yeo. Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity mri. Cerebral cortex, 28(9):3095–3114, 2018.
Dataset atl-schaefer2018 already exists. Skipping download.
All analyses complete. Results saved to: Results/PLSDA/AllReceptors
Saved heatmaps to: Figures/PLSDA/AllReceptors


---
## 5. Logistic Regression Analysis
### 5a. Forward feature selection

In [10]:
# Configuration
X_PATH_LR        = "Data/Processed/ReceptorSelection/T1wT2WReceptor_Schaefer100_df.csv"
Y_PATH_LR        = "Data/Processed/ReceptorSelection/WorkingMemory_Schaefer100.csv"
FS_SAVE_DIR      = "Results/LogisticRegression/ForwardSelection"
FS_FIGURES_DIR   = "Figures/LogisticRegression/ForwardSelection"

# Ordered list of candidates; the baseline is always added first
PREDICTOR_COLS   = ["T1w:T2w", "D1", "D2", "DAT", "NET", "NMDA", "GABAa"]
FS_TARGETS       = ["Thr_0", "Thr_0p001", "Thr_0p01", "Thr_0p1"]
BASELINE         = "T1w:T2w"
CV_FOLDS         = 5

run_forward_selection_with_baseline(
    X_path=X_PATH_LR,
    y_path=Y_PATH_LR,
    save_dir=FS_SAVE_DIR,
    figures_dir=FS_FIGURES_DIR,
    predictor_cols=PREDICTOR_COLS,
    targets=FS_TARGETS,
    baseline=BASELINE,
    cv=CV_FOLDS,
)


── Thr_0 ──
  Added D2 (AUC=0.7927, C=1.7575)
  Added D1 (AUC=0.8251, C=0.8286)
  Added GABAa (AUC=0.8487, C=0.8286)
  Added NET (AUC=0.8495, C=16.7683)
  Added NMDA (AUC=0.8495, C=0.5690)
  Added DAT (AUC=0.8495, C=0.5690)

── Thr_0p001 ──
  Added D2 (AUC=0.7950, C=24.4205)
  Added D1 (AUC=0.8259, C=3.7276)
  Added GABAa (AUC=0.8551, C=3.7276)
  Added NMDA (AUC=0.8575, C=0.8286)
  Added DAT (AUC=0.8571, C=3.7276)
  Added NET (AUC=0.8535, C=1.7575)

── Thr_0p01 ──
  Added D2 (AUC=0.8037, C=3.7276)
  Added D1 (AUC=0.8218, C=0.0281)
  Added GABAa (AUC=0.8432, C=51.7947)
  Added DAT (AUC=0.8495, C=35.5648)
  Added NMDA (AUC=0.8520, C=2.5595)
  Added NET (AUC=0.8495, C=2.5595)

── Thr_0p1 ──
  Added D2 (AUC=0.8251, C=1.2068)
  Added D1 (AUC=0.8411, C=11.5140)
  Added GABAa (AUC=0.8496, C=1.7575)
  Added DAT (AUC=0.8539, C=1.7575)
  Added NMDA (AUC=0.8571, C=1.2068)
  Added NET (AUC=0.8656, C=5.4287)
Results saved to: Results/LogisticRegression/ForwardSelection/T1w:T2w_D1_D2_DAT_NET_NMDA_G

### 5b. Model selection with spin permutation testing

In [11]:
# ── Configuration ──────────────────────────────────────────────────────────────
X_PATH_MS        = "Data/Processed/ReceptorSelection/T1wT2WReceptor_Schaefer100_df.csv"
Y_PATH_MS        = "Data/Processed/ReceptorSelection/WorkingMemory_Schaefer100.csv"
MS_SAVE_DIR      = "Results/LogisticRegression/ModelSelection"
MS_FIGURES_DIR   = "Figures/LogisticRegression/ModelSelection"

# Each inner list defines one model's feature set (M1, M2, M3, ...)
MODELS = [
    ["T1w:T2w"],
    ["T1w:T2w", "GABAa", "NMDA"],
    ["T1w:T2w", "GABAa", "NMDA", "D1", "D2", "DAT", "NET"],
]
MS_TARGETS       = ["Thr_0", "Thr_0p001", "Thr_0p01", "Thr_0p1"]
MS_N_PERM        = 1000
MS_SEED          = 1234
MS_CV_FOLDS      = 5

# ── Run ────────────────────────────────────────────────────────────────────────
results_df = run_spin_absolute_significance(
    X_path=X_PATH_MS,
    y_path=Y_PATH_MS,
    save_dir=MS_SAVE_DIR,
    figures_dir=MS_FIGURES_DIR,
    models=MODELS,
    targets=MS_TARGETS,
    cv=MS_CV_FOLDS,
    n_perm=MS_N_PERM,
    seed=MS_SEED,
)

Please cite the following papers if you are using this function:
  [primary]:
    Alexander Schaefer, Ru Kong, Evan M Gordon, Timothy O Laumann, Xi-Nian Zuo, Avram J Holmes, Simon B Eickhoff, and BT Thomas Yeo. Local-global parcellation of the human cerebral cortex from intrinsic functional connectivity mri. Cerebral cortex, 28(9):3095–3114, 2018.
Dataset atl-schaefer2018 already exists. Skipping download.

── Thr_0 ──
 M1: T1w:T2w: AUC = 0.6427, AIC = 142.56 (C = 0.0001)
 M2: T1w:T2w+GABAa+NMDA: AUC = 0.7411, AIC = 126.26 (C = 2.5595)
 M3: T1w:T2w+GABAa+NMDA+D1+D2+DAT+NET: AUC = 0.8495, AIC = 115.35 (C = 0.5690)

── Thr_0p001 ──
 M1: T1w:T2w: AUC = 0.6546, AIC = 141.14 (C = 0.0001)
 M2: T1w:T2w+GABAa+NMDA: AUC = 0.7573, AIC = 123.40 (C = 1.2068)
 M3: T1w:T2w+GABAa+NMDA+D1+D2+DAT+NET: AUC = 0.8535, AIC = 109.83 (C = 1.7575)

── Thr_0p01 ──
 M1: T1w:T2w: AUC = 0.6776, AIC = 137.70 (C = 0.0001)
 M2: T1w:T2w+GABAa+NMDA: AUC = 0.7465, AIC = 121.74 (C = 2.5595)
 M3: T1w:T2w+GABAa+NMDA+D1+D2